# Separation Judgement: Monolithic vs Modular (Parity QA)

This notebook validates module-sequential outputs against the monolithic reference on:

- `/mnt/nas02/Dataset/ca_motion_corr/caiman/2p`
- `/mnt/nas02/Dataset/ca_motion_corr/suite2p/demo`
- `/mnt/nas02/Dataset/ca_motion_corr/fiola`

It reports:

1. Dataset discovery and unsupported-file skips.
2. Boundary metrics (`max abs/rel`, `corr`, optional `SSIM`).
3. Per-module / per-dataset PASS/FAIL matrix.
4. Side-by-side visual checks (motion scores, projections, ROI overlays, traces).


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
OUT_ROOT = ROOT / "artifacts" / "qa_monolith_vs_modular"
DATASET_ROOTS = [
    Path("/mnt/nas02/Dataset/ca_motion_corr/caiman/2p"),
    Path("/mnt/nas02/Dataset/ca_motion_corr/suite2p/demo"),
    Path("/mnt/nas02/Dataset/ca_motion_corr/fiola"),
]
SUPPORTED = {".tif", ".tiff", ".avi", ".mat"}

# Toggle to re-run captures in this notebook. Reuse existing run by default.
RUN_CAPTURE = False
RUN_PICKLE_PROBE = True

# Reuse latest available run if present.
existing = sorted(OUT_ROOT.glob("*_full"))
RUN_STAMP = existing[-1].name if existing else datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_full"
RUN_DIR = OUT_ROOT / RUN_STAMP
RUN_DIR


In [ ]:
def discover_files(roots):
    supported = []
    skipped = []
    for root in roots:
        root = root.expanduser().resolve()
        if not root.exists():
            skipped.append((str(root), "missing_root"))
            continue
        for p in sorted(x for x in root.rglob("*") if x.is_file()):
            if p.suffix.lower() in SUPPORTED:
                supported.append((str(root), str(p.relative_to(root)), str(p)))
            else:
                skipped.append((str(p), f"unsupported_format:{p.suffix.lower() or '<none>'}"))
    return supported, skipped

supported, skipped = discover_files(DATASET_ROOTS)
print(f"Supported files: {len(supported)}")
for row in supported:
    print("  ", row[2])
print(f"\nSkipped files: {len(skipped)}")
for row in skipped:
    if row[1].startswith("unsupported_format"):
        print("  ", row[0], "->", row[1])


In [ ]:
def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

if RUN_CAPTURE:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    run([".venv-min1pipe/bin/python", "scripts/run_monolithic_capture.py", "--timestamp", RUN_STAMP])
    run([
        ".venv-min1pipe/bin/python",
        "scripts/run_modular_capture.py",
        "--timestamp", RUN_STAMP,
        "--mode", "parity",
        "--serialization", "none",
    ])
    if RUN_PICKLE_PROBE:
        run([
            ".venv-min1pipe/bin/python",
            "scripts/run_modular_capture.py",
            "--timestamp", RUN_STAMP,
            "--mode", "parity",
            "--serialization", "pickle",
        ])

run([".venv-min1pipe/bin/python", "scripts/compare_monolith_modular.py", "--run-dir", str(RUN_DIR)])
print("Report generated in", RUN_DIR)


In [ ]:
report = json.loads((RUN_DIR / "discrepancy_report.json").read_text())
print("Tolerance:", report["tolerance"])
print("Variants:")
for name, info in sorted(report["variants"].items()):
    print(f"  {name}: files={info['files']} pass={info['pass']}")

rows = report["rows"]
summary = report["summary"]
print("\nSummary rows:", len(summary), "Boundary rows:", len(rows))


In [ ]:
# Per-module PASS/FAIL matrix by dataset + file
module_fields = {
    "motion": ["motion__"],
    "source": ["source__"],
    "component_filtering": ["component__"],
    "calcium_deconvolution": ["deconv__"],
}

keys = sorted({(r["variant"], r["dataset_root"], r["file_rel"]) for r in rows})
matrix = []
for variant, ds, frel in keys:
    row = {"variant": variant, "dataset_root": ds, "file_rel": frel}
    subset = [r for r in rows if r["variant"] == variant and r["dataset_root"] == ds and r["file_rel"] == frel]
    for module, prefixes in module_fields.items():
        mrows = [r for r in subset if any(r["field"].startswith(p) for p in prefixes)]
        row[module] = all(x["pass"] for x in mrows) and bool(mrows)
    row["all"] = all(row[m] for m in module_fields)
    matrix.append(row)

for r in matrix:
    print(r)


In [ ]:
# Numeric aggregates
for variant in sorted({r['variant'] for r in rows}):
    vrows = [r for r in rows if r['variant'] == variant]
    max_abs = max(float(r['max_abs_err']) for r in vrows)
    max_rel = max(float(r['max_rel_err']) for r in vrows)
    mean_corr = float(np.mean([r['corrcoef'] for r in vrows]))
    print(f"{variant}: max_abs={max_abs:.6g} max_rel={max_rel:.6g} mean_corr={mean_corr:.6f}")


In [ ]:
# Load one representative file per dataset for visual checks
mono_manifest = json.loads((RUN_DIR / "manifest_monolith.json").read_text())
mod_manifest = json.loads((RUN_DIR / "manifest_modular_parity_none.json").read_text())

mono_map = {(x['dataset_root'], x['file_rel']): x for x in mono_manifest['captures']}
mod_map = {(x['dataset_root'], x['file_rel']): x for x in mod_manifest['captures']}

by_dataset = {}
for key in mono_map:
    ds, frel = key
    by_dataset.setdefault(ds, key)

fig, axes = plt.subplots(len(by_dataset), 4, figsize=(18, 4 * len(by_dataset)))
if len(by_dataset) == 1:
    axes = np.array([axes])

for i, (ds, key) in enumerate(sorted(by_dataset.items())):
    m = np.load(mono_map[key]['capture_npz'])
    b = np.load(mod_map[key]['capture_npz'])
    title = f"{Path(ds).name}/{key[1]}"

    ax = axes[i, 0]
    ax.plot(m['motion__raw_score'], label='mono_raw', lw=1.5)
    ax.plot(b['motion__raw_score'], '--', label='mod_raw', lw=1.0)
    ax.plot(m['motion__corr_score'], label='mono_corr', lw=1.5)
    ax.plot(b['motion__corr_score'], '--', label='mod_corr', lw=1.0)
    ax.set_title(f"Motion Scores\n{title}")
    ax.legend(fontsize=8)

    ax = axes[i, 1]
    diff = b['motion__imax'] - m['motion__imax']
    im = ax.imshow(diff, cmap='coolwarm')
    ax.set_title("Projection Difference (mod - mono)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axes[i, 2]
    ax.imshow(m['motion__imax'], cmap='gray')
    seeds = m['source__seedsfn'].astype(int)
    h, w = m['motion__imax'].shape
    ys, xs = np.unravel_index(seeds, (h, w))
    ax.scatter(xs, ys, s=8, c='r')
    ax.set_title("Spatial Footprints Seeds (mono)")

    ax = axes[i, 3]
    n = min(5, m['source__sigfn'].shape[0])
    for j in range(n):
        ax.plot(m['source__sigfn'][j] + 2.0 * j, c='k', lw=1.2)
        ax.plot(b['source__sigfn'][j] + 2.0 * j, c='tab:orange', lw=1.0, alpha=0.8)
    ax.set_title("Trace Overlay (black=mono, orange=mod)")

plt.tight_layout()
plt.show()


In [ ]:
# PASS/FAIL summary at required granularity
status = {}
for item in summary:
    k = (item['variant'], item['dataset_root'], item['file_rel'])
    status[k] = item['all_pass']

n_total = len(status)
n_pass = sum(1 for v in status.values() if v)
print(f"PASS {n_pass}/{n_total}")
if n_pass != n_total:
    print("Failed entries:")
    for k, ok in status.items():
        if not ok:
            print("  ", k)
